# 0. 核心思想

GSPO 全称是 Group Sequence Policy Optimization （组内序列策略优化），和 GRPO 一样保留了 组内标准化 这一步：
$$\hat{A}_i = \frac{r_i - \mu_r}{\sigma_r + \epsilon}$$


在 GRPO 中，其中的重要性采样比率 $\rho_{i,t}$ 是在 Token 级别定义的：

$$\rho_{i,t} = \frac{\pi_{\theta}(y_{i,t} \mid x, y_{i,<t})}{\pi_{\theta_{\text{old}}}(y_{i,t} \mid x, y_{i,<t})}$$


但是同一个序列中不同 token 的更新幅度可能天差地别———某个 token 的 ρ 可能是 2.8，而相邻 token 的 ρ 可能只有 0.4。这就导致：
- 引入高方差训练噪声
- 随着序列长度的增加，token 级噪声逐步累积，策略更新的不稳定性会增加
- 奖励是整个序列级别的（R(x,y)），但校正发生在 token 级别，单位不对齐

尤其在 MoE（Mixture of Experts）模型中，token 级别的梯度波动会直接影响专家路由（Router）的梯度，导致专家激活分布剧烈波动，最终触发灾难性的训练坍塌——而且往往是不可逆的。

# 1. 训练目标

## 序列级重要性采样

$$\rho_{\text{seq}}(y_i) = \exp\left(\frac{1}{|y_i|}\sum_{t=1}^{|y_i|}\log\frac{\pi_{\theta_{\text{old}}}(y_{i,t}\mid x, y_{i,<t})}{\pi_{\theta}(y_{i,t}\mid x, y_{i,<t})}\right)$$
等价于：
$$\rho_{\text{seq}}(y_i) = \left(\frac{\pi_{\theta_{\text{old}}}(y_i \mid x)}{\pi_{\theta}(y_i \mid x)}\right)^{1/|y_i|}$$


- $\frac{\pi_{\theta}(y_{i,t} \mid x, y_{i,<t})}{\pi_{\theta_{\text{old}}}(y_{i,t} \mid x, y_{i,<t})} = \frac{1}{\rho_{i,t}}$：单 token 概率比，与 $\rho_{i,t}$ 互为倒数，用旧策略除以新策略

- $\frac{1}{|y_i|} \sum_{t=1}^{|y_i|} \log \frac{\pi_{\theta_{\text{old}}}}{\pi_{\theta}}$：对所有 token 的 log-ratio 取算术平均。log 空间做平均，本质上就是在算几何平均（概率连乘后取平均）

- $\exp(几何平均概率比)$：平均 log-ratio 的指数，等于几何平均概率比，即 $\rho_{i,t}$ 的几何平均的倒数


GSPO 的 $\rho_{\text{seq}}$ 对一个序列中所有 token 的对数概率比率取平均，然后用这个统一的系数来缩放该序列中所有 token 的梯度。这就像给全班同学打了一个统一的"班级系数"，而不是每个人乘一个随机系数。

**为什么对概率比取对数再取均值？**

- 算术平均：$\displaystyle \frac{a + b + c}{3}$：把数**加起来**再平分
- 几何平均：$\displaystyle (a \times b \times c)^{1/3}$：把数**乘起来**再开方

概率的乘法关系在对数空间下才变为加法关系，概率比天然是**乘性累积**的，故使用几何平均来表示序列中所有 token 的概率比。

**这也是为什么 GSPO 对长度变化具有天然的鲁棒性：** <br>
例如：
- 序列 A（短）： [0.5, 2.0] ：一个 token 概率翻倍，一个 token 概率减半
- 序列 B（长）： [0.5, 2.0, 0.5, 2.0] ： A 的两倍长度，模式相同

| 方法 | 序列 A | 序列 B |
|------|--------|--------|
| 算术平均 | $(0.5 + 2.0) / 2 = 1.25$ | $(0.5 + 2.0 + 0.5 + 2.0) / 4 = 1.25$ |
| 几何平均 | $(0.5 \times 2.0)^{1/2} = 1.0$ | $(0.5 \times 2.0 \times 0.5 \times 2.0)^{1/4} = 1.0$ |

故几何平均使长度鲁棒：A 和 B 的几何平均相等，长度翻倍不影响结果——$\displaystyle \left(\prod_{i=1}^{n} \frac{\pi_\theta(o_i|q,o_{<i})}{\pi_{\text{ref}}(o_i|q,o_{<i})}\right)^{1/n}$ 中的指数 $1/n$ 抵消了长度的影响。


## 完整目标函数

$$L_{GSPO}(\theta) = -\frac{1}{G} \sum_{i=1}^{G} \frac{1}{|y_i|} \sum_{t=1}^{|y_i|} \min\left(\rho_{seq}(y_i) \cdot \hat{A}_i, \text{clip}(\rho_{seq}(y_i), 1-\epsilon, 1+\epsilon) \cdot \hat{A}_i\right)$$


## 与 GRPO 的对比

| 维度 | GRPO | GSPO |
|------|------|------|
| 重要性比率 | $\rho_{i,t}$（每个 token 不同） | $\rho_{\text{seq}}(y_i)$（同序列所有 token 相同） |
| Clip 作用对象 | 逐 token 裁剪 | 整序列裁剪 |
| Clip 超参数 $\epsilon$ | 0.1 ~ 0.2 | 3e-4 ~ 4e-4（远更紧凑） |
| 优势函数 | 组内标准化（不变） | 组内标准化（不变） |
| 梯度一致性 | 同序列 token 更新幅度各异 | 同序列 token 更新幅度统一 |
| KL 约束 | 需要显式 KL 惩罚项 | Clip 本身足够约束（可省略 KL） |


GSPO 的 Clip 超参数 $\epsilon$ 更小的原因：

GSPO 对序列级 $\rho_{\text{seq}}$ 裁剪，$\rho_{\text{seq}}$ 是对数均值取指数的结果，其波动范围天然远小于单 token 的 $\rho_{i,t}$，所以需要更紧凑的 $\epsilon$ 来精细控制。


# 2. 补充

**GSPO所有token重要性比率相等，为什么是有效的，这是否有副作用？**

为什么有效：
- 奖励信号本身就是序列级的，不是 Token 级的
- Token 级 $\rho$ 的差异大多是噪声，不是信号
- MoE 架构的特殊脆弱性：

    MoE 模型中每个 token 激活不同的专家组合。如果 token 间 $\rho$ 差异大 $\rightarrow$ token 间梯度差异大 $\rightarrow$ 不同 token 对 Router 的梯度信号互相矛盾 $\rightarrow$ 专家激活分布震荡 $\rightarrow$ 训练坍塌。GSPO 统一了所有 token 的更新幅度，Router 收到一致的梯度信号，从根本上消除了这个风险。

- 更紧的 Clip 填补了正则化缺口

    注意对比表中的关键细节：GRPO 的 $\epsilon \in [0.1, 0.2]$，GSPO 的 $\epsilon \in [3 \times 10^{-4}, 4 \times 10^{-4}]$ —— 差了约 500 倍。因为 $\rho_{\text{seq}}$ 是几何平均，波动远小于单 token 的 $\rho_{i,t}$，可以用极紧的 clip 精细控制每一步的更新幅度。这使得 GSPO 甚至可以**省略显式的 KL 惩罚项**，clip 本身就足够约束。

副作用：
- 信用分配粒度变粗

    如果一个回答整体是对的但中间有一句胡说八道，GSPO 无法单独抑制那句错误——整条序列一起被增强或一起被抑制。但实际上，LLM 的奖励函数通常是结果导向的（答案对不对、格式好不好），极少有 Token 级的细粒度反馈，所以这个理论上的损失在实践中并不显著。

- 长尾（一条分布中数量少但差异大的那部分） token 被过度平均

    假设一条序列中 90% 的 token 是高频常见词（"的"、"是"、"the"），$\rho \approx 1$，只有最后几个关键推理 token 的 $\rho$ 偏离很大。几何平均会把这些关键 token 的差异稀释掉，导致对真正重要的 token 更新不够激进。


这是 GSPO 最实际的 trade-off： **用信用分配精度换训练稳定性** 。对于 DeepSeek-R1、Qwen 这类千亿参数 MoE 模型，稳定性是硬需求，所以这个 trade-off 是值得的。但对于小模型或非 MoE 架构，GRPO 的 Token 级更新可能仍有优势。

**为什么 GSPO 稳定了 MoE 模型的训练？**

MoE 模型，如 Qwen3-30B-A3B（30B 总参数、每次激活 3B 的 MoE 架构）

MoE 模型中有一个 Router（路由器） 网络，负责决定**每个 token** 由哪些专家处理。Router 的梯度直接受到每个 token 更新幅度的影响。

- 在 GRPO 中，不同 token 的 $\rho_{i,t}$ 差异很大 → Router 收到的梯度信号忽大忽小 → 专家激活分布剧烈波动 → 某些专家可能逐渐"死亡"（从不被路由到）或某些专家被过度使用 → 不可逆的训练坍塌。

- 在 GSPO 中，所有 token 共享统一的 $\rho_{\text{seq}}$ → Router 收到的梯度信号稳定一致 → 专家激活分布平滑 → 即使不使用 Routing Replay 等复杂技巧，训练也能保持稳定。


**GRPO 与 GSPO 的梯度差异：**

GRPO：
$$L = -\frac{1}{G} \sum_{i} \frac{1}{|y_i|} \sum_{t} \min\left( \rho_{i,t} \hat{A}_i, \text{clip}(\dots) \right)$$

GSPO：
$$L = -\frac{1}{G} \sum_{i} \frac{1}{|y_i|} \sum_{t} \min\left( \rho_{\text{seq}}(y_i) \hat{A}_i, \text{clip}(\dots) \right)$$

唯一的差别就是红框里那个量——但这一处差别决定了梯度的走向：

梯度对比（假设 $\hat{A}^i > 0$，未触发 clip）

GRPO：token $t$ 的梯度 $\propto \rho_{i,t} \cdot \frac{\partial}{\partial \theta} \log \pi_\theta(y_{i,t})$

GSPO：token $t$ 的梯度 $\propto \rho_{\text{seq}} \cdot \frac{\partial}{\partial \theta} \log \pi_\theta(y_{i,t})$

| 特性 | GRPO | GSPO |
|:---|:---|:---|
| 每个 token 的乘数 | 各不相同（1.2、2.8、0.4...） | 全部相同（比如都是 1.1） |
| 同一条序列内 token 间梯度幅度 | 有人迈大步，有人迈小步 | 所有人迈一样的步长 |


# 3. Reference

[1] [GRPO/GSPO：组内相对策略优化与奖励函数设计](https://haozhe-xing.github.io/agent_learning/zh/chapter_agentic_rl/05_grpo.html) <br>